# Title

panda tables probably would've been a good idea


In [477]:
import glob
import re
import math
import random
import pprint as pp
import numpy as np

import k3d
from k3d import platonic

PHI = 1.618033988749895

# a = [1,2,3]
# b = [*a, 4,5,6] # ... nice, except when it do special idiomatic thing

## Constants

### Field

In [478]:
F_X = 325.611*2
F_Y = 158.844*2
F_Z = 180
FIELD_DIMS = [F_X/2., F_Y/2., F_Z, 1]
TO_BLUE = [-(F_X/2), -(F_Y/2), 0, 1]       # from origin (but flipped for translation)
TO_ORIGIN = [-TO_BLUE[0],-TO_BLUE[1],0,1]  # from blue

# both @ and * are doing the same thing...?
xf_blue = np.multiply(np.matrix([-1,1,-1,-1], dtype=np.float32).T @ np.matrix([1,1,1,1]), np.eye(4))
xf_blue[:,3] = np.matrix(TO_ORIGIN).T
xf_blue

matrix([[ -1.   ,  -0.   ,  -0.   , 325.611],
        [  0.   ,   1.   ,   0.   , 158.844],
        [ -0.   ,  -0.   ,  -1.   ,   0.   ],
        [ -0.   ,  -0.   ,  -0.   ,   1.   ]])

### Targets

Icosahedrons that should fit a ball in their interior

In [479]:
T_X = 182.11
T_Y = 158.844
TARGET_SIZE = 5.91; # 5.91/PHI;
TARGET_BLUE = [T_X, T_Y, 72,1]
target_blue = platonic.Icosahedron(origin=TARGET_BLUE[0:3], size=(5.91/PHI))
target_blue.mesh.color = 0xFF0000
TARGET_RED = [(F_X-T_X),(F_Y-T_Y),72,1]
target_red = platonic.Icosahedron(origin=TARGET_RED[0:3], size=(5.91/PHI))
target_red.mesh.color = 0x0000FF

### Trajectory Calculations

- started with reasonable heights. `> 4.0m` too high. arcs take more time.
- `45cm` expected turret height offsets potential energy
- Calculated V_{z range} offset from turret & hub

In [484]:
GP_M = 2.10 # to 2.27
GP_R = 5.91/2

M_TO_IN = (100/2.54)
H_MIN = 2.2
H_MAX = 4.0
h_range = np.matrix([H_MIN,H_MAX])
h_range_in = h_range * M_TO_IN

G = -4.9
H_TURRET = 0.45
H_HUB = 72/M_TO_IN
TH_MIN = 45*(math.pi/180)
TH_MAX = 79*(math.pi/180)
k_range = GP_M * (h_range - H_TURRET) # kg * m
k_range_hub = GP_M * (h_range - H_HUB) # kg * m
vz_range = np.sqrt(abs(2*G*(h_range - H_TURRET))) # from turret
vz_range_hub = np.sqrt(abs(2*G*(h_range - H_HUB))) # from turret
vz_range_hub
# [H_MIN, H_MAX] * 1.0
#* M_TO_IN
# = 4.5 * M_TO_IN
# H_MIN = 2.2* 

t_range_hub_half = np.sqrt(2*(h_range - H_HUB)/-G)
t_range_hub = 2 * t_range_hub_half
vz_avg_t2h = (vz_range_hub + vz_range)/2
t_range_t2h = [H_HUB-H_TURRET,H_HUB-H_TURRET] / VZ_AVG_T2H

In [485]:
pp.pprint([vz_avg_t2h, t_range_t2h])

[matrix([[3.02427356, 5.25554354]]), matrix([[0.45591114, 0.26235155]])]


In [487]:
dict(H_MIN=H_MIN, H_MAX=H_MAX, h_range=h_range, h_range_in=h_range_in, 
     H_TURRET=H_TURRET, H_HUB=H_HUB, 
     k_range=k_range, k_range_hub=k_range_hub, t_range_hub=t_range_hub,
     vz_range=vz_range, vz_range_hub=vz_range_hub)


{'H_MIN': 2.2,
 'H_MAX': 4.0,
 'h_range': matrix([[2.2, 4. ]]),
 'h_range_in': matrix([[ 86.61417323, 157.48031496]]),
 'H_TURRET': 0.45,
 'H_HUB': 1.8288,
 'k_range': matrix([[3.675, 7.455]]),
 'k_range_hub': matrix([[0.77952, 4.55952]]),
 't_range_hub': matrix([[0.77848623, 1.88276826]]),
 'vz_range': matrix([[4.14125585, 5.89830484]]),
 'vz_range_hub': matrix([[1.90729127, 4.61278224]])}

In [488]:
4.414 /0.70219398

6.286012306741792

In [491]:
vx_th_max = (1/math.tan(TH_MAX))*VZ_RANGE
v_th_max = VZ_RANGE/math.sin(TH_MAX)
vx_th_min = (1/math.tan(TH_MIN))*VZ_RANGE
v_th_min = VZ_RANGE/math.sin(TH_MIN)

vx_th_max_hub = (1/math.tan(TH_MAX))*vz_range_hub
v_th_max_hub = vz_range_hub/math.sin(TH_MAX)
vx_th_min_hub = (1/math.tan(TH_MIN))*vz_range_hub
v_th_min_hub = vz_range_hub/math.sin(TH_MIN)

# then calc time to hub height -> gets a symmetric parabola

["Vx min/max (θ max)",vx_th_max,  # 2.2m
 "V  min/max (θ max)",v_th_max ,  # 4.0m
 "Vx min/max (θ min)",vx_th_min,  # 2.2m
 "V  min/max (θ min)",v_th_min]   # 4.0m

['Vx min/max (θ max)',
 matrix([[0.80497859, 1.14651432]]),
 'V  min/max (θ max)',
 matrix([[4.21876647, 6.00870161]]),
 'Vx min/max (θ min)',
 matrix([[4.14125585, 5.89830484]]),
 'V  min/max (θ min)',
 matrix([[5.85662019, 8.3414627 ]])]

In [492]:

pp.pprint(
[vx_th_min,
v_th_min,
vx_th_min_hub,
v_th_min_hub,
vx_th_max,
v_th_max,
vx_th_max_hub,
v_th_max_hub])


[matrix([[4.14125585, 5.89830484]]),
 matrix([[5.85662019, 8.3414627 ]]),
 matrix([[1.90729127, 4.61278224]]),
 matrix([[2.69731719, 6.52345921]]),
 matrix([[0.80497859, 1.14651432]]),
 matrix([[4.21876647, 6.00870161]]),
 matrix([[0.37073987, 0.89663404]]),
 matrix([[1.94298946, 4.69911828]])]


In [493]:
["v min/max @ theta max", 
 np.multiply(vx_th_max,vx_th_max), 
 np.multiply(vz_range,vz_range),
 np.multiply(v_th_max,v_th_max),
 "check",
 np.multiply(v_th_max,v_th_max) - np.multiply(vx_th_max,vx_th_max),
 "v min/max @ theta min",
 np.multiply(vx_th_min,vx_th_min), 
 np.multiply(vz_range,vz_range),
 np.multiply(v_th_min,v_th_min),
 "check",
 np.multiply(v_th_min,v_th_min) - np.multiply(vx_th_min,vx_th_min)]

['v min/max @ theta max',
 matrix([[0.64799053, 1.31449508]]),
 matrix([[17.15, 34.79]]),
 matrix([[17.79799053, 36.10449508]]),
 'check',
 matrix([[17.15, 34.79]]),
 'v min/max @ theta min',
 matrix([[17.15, 34.79]]),
 matrix([[17.15, 34.79]]),
 matrix([[34.3 , 69.58]]),
 'check',
 matrix([[17.15, 34.79]])]

In [494]:
def stl_props(f, name=None, group=None, **kwargs):
    rxgroup = r'stl/([^_]+)_.*.stl'
    rxname = r'stl/[^_]+_(.*).stl'
    name = (name or re.sub(rxname, r'\1', f))
    group = (group or re.sub(rxgroup, r'\1', f))
    return dict(name=name, group=group, **kwargs)
np.column_stack
# flat_shading = True, shininess = 50.0, compression_level = 0
def stl_read(f:str, color = 0x0000ff, wireframe = False, name = None, group = None, custom_data = None):
    with open(f, 'r') as stl:
        data = stl.read()
    return k3d.stl(data, color=color, wireframe=wireframe, name=name, group=group, custom_data=custom_data)

In [495]:
stl_blue = [stl_read(f, **stl_props(f, color=0x000066)) for f in glob.glob('stl/blue_*.stl') ]
stl_red = [stl_read(f, **stl_props(f), color=0x660000) for f in glob.glob('stl/red_*.stl') ]
stl_neutral = [stl_read(f, **stl_props(f, color=0xaaaaaa)) for f in glob.glob('stl/neutral_*.stl') ]
stl_all = (stl_blue + stl_red + stl_neutral)

for s in stl_all: s.model_matrix = xf_blue

/usr/local/lib/python3.12/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(


In [496]:
plot = k3d.plot()
# for s in (stl_blue + stl_red + stl_neutral): plot += s
for s in stl_all: plot += s
plot += target_blue.mesh
plot += target_red.mesh

plot.display()

Output()

## Misc

In [497]:
def randomblobs(n=10):
    blobs = []
    for i in range(n): 
        rndpoint = [F_X*random.gauss(0.5,0.25),
                    F_Y*random.gauss(0.5,0.25), 
                    F_Z*random.gauss(0.75,0.125)]
        b = platonic.Icosahedron(origin=rndpoint, size=(5/PHI))
        
        b.mesh.color = 0xFFFF00
        blobs.append(b)
    return blobs

### Updating live plots

- `display()` re-renders the plot in a new cell
- `render()` doesn't work across cells unless the plot is in "callback" mode
- better info in [time series](https://k3d-jupyter.org/user/time.html)
- can pre-generate time-series data for animation (like in [hydrogen_orbitals.ipynb](https://github.com/K3D-tools/K3D-jupyter/blob/main/examples/hydrogen_orbitals.ipynb))

In [498]:
# for b in randomblobs(100): plot += b.mesh
# plot.render() 

In [499]:
bb = randomblobs(5)
type(bb[0].mesh)

# bb[0].mesh.set_trait(color=0xFFFF00)

k3d.objects.geometry.Mesh

In [425]:
# rndpoint = [F_X*random.gauss(0.5,0.25),
#                     F_Y*random.gauss(0.5,0.25), 
#                     F_Z*random.gauss(0.75,0.125)]
# rndpoint